<a href="https://colab.research.google.com/github/fahadansari1907-spec/Netflix/blob/main/Netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Movie Recommendation Engine

## Project Overview

This notebook demonstrates the process of building a movie recommendation engine using the Netflix prize dataset. The goal is to predict how a user would rate a movie they haven't seen yet, and then recommend movies with the highest predicted ratings.

We will cover the following key stages:
1.  **Data Loading and Initial Exploration**: Importing necessary libraries and understanding the raw dataset.
2.  **Data Preprocessing**: Cleaning and transforming the data for model training.
3.  **Model Training (SVD)**: Utilizing the Singular Value Decomposition (SVD) algorithm from the surprise library to build a collaborative filtering model.
4.  **Recommendation Generation**: Predicting ratings for unseen movies and generating top recommendations for a specific user.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Data Loading and Initial Exploration

This section handles the loading of the main Netflix dataset and initial inspection to understand its structure, identify missing values, and get a preliminary sense of the data distribution.

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Netflix/Copy of combined_data_1.txt.zip",header=None,names=["Cust_ID","Ratings"],usecols=[0,1])

### Dataset Overview

The primary dataset combined_data_1.txt.zip contains customer IDs and their ratings. The format is a bit unconventional where movie IDs are interleaved. The initial loading treats the first column as Cust_ID and the second as Ratings. The first row of each movie block contains only the Movie_ID followed by a colon, which is handled in subsequent preprocessing steps.

In [ ]:
df

,Cust_ID,Ratings
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0
...,...,...
24058258,2591364,2.0
24058259,1791000,2.0
24058260,512536,5.0
24058261,988963,3.0


The df.head() output shows how the raw data is loaded, with Cust_ID containing movie identifiers (like '1:') at the start of each movie block, and actual customer IDs mixed with ratings. The presence of '1:' also leads to NaN values in the Ratings column for those rows, indicating a movie header rather than a rating entry.

In [ ]:
df.dtypes

,0
Cust_ID,object
Ratings,float64


### Data Types and Missing Values

Checking data types helps ensure columns are in the correct format for analysis. Identifying null values is crucial for deciding how to clean or impute them.

As observed, Cust_ID is object due to the mixed integer and string ('X:') values, and Ratings is float64 but contains NaNs where movie IDs are present.

In [ ]:
df.isnull().sum()

,0
Cust_ID,0
Ratings,4499


The sum of null values confirms the presence of NaNs in the Ratings column, corresponding to the movie ID header rows.

In [ ]:
movie_count=df.isnull().sum()["Ratings"]

In [ ]:
total_unique=df['Cust_ID'].nunique()


In [ ]:
cust_count=total_unique-movie_count

In [ ]:
ratings_count=df['Ratings'].count()-movie_count

In [ ]:
stars_count=df['Ratings'].value_counts()
stars_count

,count
Ratings,
4.0,8085741
3.0,6904181
5.0,5506583
2.0,2439073
1.0,1118186


### Rating Distribution

This shows the count of each rating (1 to 5 stars) across the dataset, giving insight into how users rate movies generally. We can see that 4-star and 3-star ratings are the most frequent.

In [ ]:
movie_id=None
movie_list=[]
for customer in df["Cust_ID"]:
  if ":" in customer:
    movie_id=int(customer.replace(":",""))
  movie_list.append(movie_id)

## 2. Data Preprocessing

This section focuses on cleaning and transforming the raw data into a suitable format for the recommendation model. This involves extracting movie IDs, handling missing values, and filtering out infrequent movies or users to reduce noise and improve model performance.

In [ ]:
movie_list

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


### Extracting Movie IDs

The original dataset interleaves movie IDs with customer ratings. This loop iterates through the 'Cust_ID' column to identify rows containing a movie identifier (e.g., '1:') and assigns the corresponding Movie_ID to all subsequent rating entries until a new movie ID is encountered. The extracted Movie_ID is stored in movie_list.

In [ ]:
df["Movie_ID"]=movie_list

### Cleaning and Filtering Data

After extracting Movie_ID into a new column, rows corresponding to movie headers (where Ratings are NaN) are dropped. Cust_ID is then converted to an integer type. Finally, the dataset is filtered to include only movies and customers that have a sufficient number of ratings, which helps to remove sparse data and focus on more active users and popular movies.

In [ ]:
df.dropna(inplace=True)

In [ ]:
df["Cust_ID"]=df['Cust_ID'].astype("int")
df

,Cust_ID,Ratings,Movie_ID
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1
...,...,...,...
24058258,2591364,2.0,4499
24058259,1791000,2.0,4499
24058260,512536,5.0,4499
24058261,988963,3.0,4499


The df DataFrame now contains Cust_ID, Ratings, and Movie_ID in a clean, structured format, ready for further processing. The number of rows is reduced after dropping NaNs.

In [ ]:
movie_rat_counts=df["Movie_ID"].value_counts()
movie_rat_counts

,count
Movie_ID,
1905,193941
2152,162597
3860,160454
4432,156183
571,154832
...,...
4294,44
915,43
3656,42


### Movie and Customer Activity Analysis

movie_rat_counts provides the number of ratings each movie has received. This is used to identify and potentially filter out movies with very few ratings, as they might not have enough data to generate reliable recommendations.

In [ ]:
bench_marks1=round(movie_rat_counts.quantile(0.6))
bench_marks1

908

The bench_marks1 is a threshold (60th percentile) for the minimum number of ratings a movie must have to be included in the filtered dataset. Movies with ratings counts below this benchmark will be dropped.

In [ ]:
drop_movie_index=movie_rat_counts[movie_rat_counts<bench_marks1].index
drop_movie_index

Index([1598, 1647, 1733, 4099, 1616, 1446,  263,  160, 4259,  322,
       ...
       1858, 4035, 3693, 2805,  820, 4294,  915, 3656, 4338, 4362],
      dtype='int64', name='Movie_ID', length=2699)

This Index contains the Movie_IDs of movies that fall below bench_marks1 and will be excluded from the dataset.

In [ ]:
cust_rat_index=df["Cust_ID"].value_counts()
cust_rat_index

,count
Cust_ID,
305344,4467
387418,4422
2439493,4195
1664010,4019
2118461,3769
...,...
739382,1
1504246,1
1244849,1


cust_rat_index similarly provides the number of ratings each customer has given. This is used to filter out inactive users who have given very few ratings.

In [ ]:
bench_mark2=round(cust_rat_index.quantile(0.6))
bench_mark2

36

The bench_mark2 is a threshold (60th percentile) for the minimum number of ratings a customer must have given. Customers with ratings counts below this benchmark will be dropped.

In [ ]:
drop_cust_rat_index=cust_rat_index[cust_rat_index<bench_mark2]
drop_cust_rat_index

,count
Cust_ID,
1804808,35
2619850,35
1023928,35
1475023,35
2569169,35
...,...
739382,1
1504246,1
1244849,1


This Series contains the Cust_IDs of customers who fall below bench_mark2 and will be excluded from the dataset.

In [ ]:
df=df[~df['Movie_ID'].isin(drop_movie_index)]
df=df[~df["Cust_ID"].isin(drop_cust_rat_index)]

The dataset df is now filtered by removing movies and customers that do not meet the minimum rating thresholds. This step significantly reduces the dataset size and improves the quality of the data for the recommendation model by focusing on more relevant interactions.

In [ ]:
df

,Cust_ID,Ratings,Movie_ID
695,1025579,4.0,3
696,712664,5.0,3
697,1331154,4.0,3
698,2632461,3.0,3
699,44937,5.0,3
...,...,...,...
24056844,267802,4.0,4496
24056845,1559566,3.0,4496
24056846,293198,3.0,4496
24056847,70814,2.0,4496


The df DataFrame after filtering shows a reduced number of rows and contains only active users and popular movies. This cleaned dataset is now suitable for building the recommendation model.

In [ ]:
stars_count=df['Ratings'].value_counts()
stars_count

,count
Ratings,
4.0,7878811
3.0,6646522
5.0,5369050
2.0,2307706
1.0,1015771


The updated rating counts after filtering out infrequent movies and customers. The distribution generally remains similar, but the total counts are lower.

In [ ]:
movie_title = pd.read_csv("/content/drive/MyDrive/Netflix/movies.csv", encoding='ISO-8859-1', header=0)
movie_title.columns = ['Movie_ID', 'Year', 'Name']
movie_title['Movie_ID'] = movie_title['Movie_ID'].astype(int)

### Loading Movie Titles

To provide human-readable recommendations, we need a separate dataset that maps Movie_ID to actual movie titles. The movies.csv file serves this purpose. It is loaded, and its columns are renamed for clarity. The Movie_ID column is cast to an integer type to ensure consistency with our main df.

In [ ]:
movie_title

,Movie_ID,Year,Name
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
27273,131254,Kein Bund fÃ¼r's Leben (2007),Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy
27275,131258,The Pirates (2014),Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed)


Displaying the movie_title DataFrame confirms its structure with Movie_ID, Year, and Name (which originally might contain genres but is used for the movie title). This DataFrame will be joined with the recommendations to show movie names instead of just IDs.

In [ ]:
! pip install scikit_surprise

## 3. Building the Recommendation Model (SVD)

This section focuses on setting up and training a collaborative filtering model using the surprise library. We will use the Singular Value Decomposition (SVD) algorithm, a popular matrix factorization technique, to predict user ratings.

In [ ]:
from surprise import Reader,Dataset,SVD
from surprise.model_selection import cross_validate

### Surprise Library Setup

The surprise library is specifically designed for building and analyzing recommender systems. We import Reader to parse the dataset, Dataset to load it, SVD as our chosen model, and cross_validate for model evaluation.

In [ ]:
reader=Reader()

### Data Preparation for Surprise

The Reader object defines the rating scale used in the dataset (implicit here as default is 1-5). The Dataset.load_from_df method is then used to load our preprocessed df into a format compatible with surprise, specifying the columns for Movie_ID, Cust_ID, and Ratings. For demonstration, only the first 100,000 entries are used to speed up computation.

In [ ]:
data=Dataset.load_from_df(df[["Movie_ID","Cust_ID","Ratings"]][:100000],reader)

In [ ]:
data


This output confirms that the data has been successfully loaded into a surprise.dataset.DatasetAutoFolds object, ready for model training.

In [ ]:
model=SVD()

### SVD Model Initialization and Evaluation

An instance of the SVD algorithm is created. SVD is a powerful matrix factorization technique that decomposes the user-item interaction matrix into lower-dimensional matrices, capturing latent factors that explain user preferences and item characteristics.

In [ ]:
cross_validate(model,data,measures=["RMSE"],cv=3)

{'test_rmse': array([1.04289125, 1.04753926, 1.04411022]),
 'fit_time': (1.5324628353118896, 1.5187928676605225, 1.901914119720459),
 'test_time': (0.22306489944458008, 0.1738746166229248, 0.7931699752807617)}

The cross_validate function evaluates the SVD model's performance using cross-validation. It splits the data into multiple folds, trains the model on some folds, and tests on others. We are measuring RMSE (Root Mean Squared Error), which is a common metric for evaluating the accuracy of predicted ratings. A lower RMSE indicates better prediction accuracy.

In [ ]:
user_1025579=df[df["Cust_ID"]==1025579]
user_1025579

,Cust_ID,Ratings,Movie_ID
695,1025579,4.0,3
3013927,1025579,4.0,571
3533410,1025579,5.0,677
4053784,1025579,4.0,788
4146733,1025579,4.0,798
4274388,1025579,3.0,819
4856581,1025579,3.0,985
6881597,1025579,3.0,1367
8879190,1025579,4.0,1770
11395230,1025579,1.0,2178


## 4. Recommendation Generation

This section demonstrates how to generate recommendations for a specific user. It involves identifying movies a user has already rated, predicting ratings for unseen movies, and then presenting the top recommended movies.

In [ ]:
dummy_title=movie_title.copy()

### User-Specific Recommendations

We first select a specific user (Cust_ID 1025579 in this case) and inspect their past ratings. This helps in understanding what movies they have already seen and rated, which will be excluded from recommendations.

In [ ]:
drop_movie_index

Index([1598, 1647, 1733, 4099, 1616, 1446,  263,  160, 4259,  322,
       ...
       1858, 4035, 3693, 2805,  820, 4294,  915, 3656, 4338, 4362],
      dtype='int64', name='Movie_ID', length=2699)

A copy of the movie_title DataFrame (dummy_title) is created to hold the movie information for recommendation generation. This ensures that the original movie_title DataFrame remains unaltered.

In [ ]:
dummy_title=dummy_title[~dummy_title['Movie_ID'].isin(drop_movie_index)]

The dummy_title DataFrame is filtered to remove movies that were identified as having insufficient ratings during the preprocessing step (drop_movie_index). This ensures that recommendations are only generated for movies with enough data.

In [ ]:
dummy_title

,Movie_ID,Year,Name
2,3,Grumpier Old Men (1995),Comedy|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
7,8,Tom and Huck (1995),Adventure|Children
15,16,Casino (1995),Crime|Drama
...,...,...,...
27273,131254,Kein Bund fÃ¼r's Leben (2007),Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy
27275,131258,The Pirates (2014),Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed)


The dummy_title DataFrame, after filtering, now contains only the movies that are considered valid for recommendation based on our earlier thresholds. The initial 'movieId' row is then dropped, as it's a header from the movies.csv file and not an actual movie entry.

### Predicting Ratings for Unseen Movies

For each movie in the dummy_title DataFrame, the trained SVD model predicts the rating that the selected user (1025579) would give to that movie. These predicted ratings are stored in the est_rating list.

In [ ]:
est_rating=[]
for movie in dummy_title["Movie_ID"]:
  rating=model.predict(1025579,movie).est
  est_rating.append(rating)

The est_rating list contains the predicted ratings for all movies in dummy_title for the selected user. Currently, it appears to contain identical predicted ratings for all movies, which might indicate an issue with the training data (due to the small subset [:100000] used for Dataset.load_from_df) or the model's ability to differentiate preferences on this specific subset. For a real-world scenario, a larger training dataset would yield more varied and accurate predictions.

In [ ]:
est_rating

[np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float64(3.5486672566637165),
 np.float6

### Displaying Top Recommendations

The predicted ratings are added as a new column to the dummy_title DataFrame. The DataFrame is then sorted by these predicted ratings in descending order to identify the movies with the highest estimated ratings for the user.

In [ ]:
dummy_title['Rating'] = est_rating
dummy_title

,Movie_ID,Year,Name,Rating
2,3,Grumpier Old Men (1995),Comedy|Romance,3.548667
4,5,Father of the Bride Part II (1995),Comedy,3.548667
5,6,Heat (1995),Action|Crime|Thriller,3.548667
7,8,Tom and Huck (1995),Adventure|Children,3.548667
15,16,Casino (1995),Crime|Drama,3.548667
...,...,...,...,...
27273,131254,Kein Bund fÃ¼r's Leben (2007),Comedy,3.548667
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy,3.548667
27275,131258,The Pirates (2014),Adventure,3.548667
27276,131260,Rentun Ruusu (2001),(no genres listed),3.548667


The dummy_title DataFrame now includes the Rating column (predicted rating for the user). This DataFrame is then sorted to find the highest-rated movies for the target user.

In [ ]:
dummy_title=dummy_title.sort_values("Rating",ascending=False)

In [ ]:
dummy_title.head(5)

,Movie_ID,Year,Name,Rating
17970,90154,"Caller, The (2011)",Horror|Mystery|Thriller,3.781756
9461,27746,Ginger Snaps: Unleashed (2004),Horror|Thriller,3.777671
16535,83540,"Silence of the Sea, The (Le silence de la mer)...",Drama|Romance|War,3.758950
2689,2775,Head On (1998),Drama,3.745664
7021,7133,Metalstorm: The Destruction of Jared-Syn (1983),Action|Adventure|Sci-Fi,3.727081
